# 21 · Descriptors & `property`

**Topics covered:** `property()` function · `@property` getter / setter / deleter · validation in setters · name validator exercise · real descriptor protocol (`__get__`, `__set__`)

## 1. Idea
A **descriptor** controls how an attribute is **read, written and deleted**.
Python's built-in `property` is the easiest descriptor: it lets you run **getter / setter / deleter** code while the user still writes `obj.age`.

## 2. Using the `property()` function

In [1]:
class Student:
    def getAge(self):
        print("Getter is called")
        return self._age

    def setAge(self, age):
        print("Setter is called")
        self._age = age

    def delAge(self):
        print("Deleter is called")
        del self._age

    age = property(getAge, setAge, delAge)    # descriptor using property

s = Student()
s.age = 25          # -> setter
print(s.age)        # -> getter
del s.age           # -> deleter

Setter is called
Getter is called
25
Deleter is called


## 3. Using the `@property` decorators (preferred)

In [2]:
class person:
    @property
    def age(self):
        print("Getter is called")
        return self._age

    @age.setter
    def age(self, age):
        print("Setter is called")
        self._age = age

    @age.deleter
    def age(self):
        print("Deleter is called")
        del self._age

s = person()
s.age = 25
print(s.age)
del s.age

Setter is called
Getter is called
25
Deleter is called


## 4. Validation inside the setter
Age must be between 18 and 100, otherwise store `None`.

In [3]:
class person:
    @property
    def age(self):
        print("Getter is called")
        return self._age

    @age.setter
    def age(self, age):
        print("Setter is called")
        if age in range(18, 101):
            self._age = age
        else:
            print("Invalid")
            self._age = None

    @age.deleter
    def age(self):
        print("Deleter is called")
        del self._age

s = person()
s.age = 25
print(s.age)
s.age = 5            # invalid
print(s.age)
del s.age

Setter is called
Getter is called
25
Setter is called
Invalid
Getter is called
None
Deleter is called


## 5. Exercise – descriptor `name`
*Create a descriptor `name` to validate the name of a person: it must be a **string** and its **length must not be more than 8**.*

In [4]:
class person:
    @property
    def name(self):
        print("Getter is called")
        return self._name

    @name.setter
    def name(self, name):
        print("Setter is called")
        if isinstance(name, str) and len(name) <= 8:
            self._name = name
        else:
            print("Invalid name (must be str and max 8 characters)")
            self._name = None

    @name.deleter
    def name(self):
        print("Deleter is called")
        del self._name

p = person()
p.name = "Shubham"
print(p.name)
p.name = "Shubham Jadhav"     # too long
print(p.name)
p.name = 123                  # not a string
print(p.name)
del p.name

Setter is called
Getter is called
Shubham
Setter is called
Invalid name (must be str and max 8 characters)
Getter is called
None
Setter is called
Invalid name (must be str and max 8 characters)
Getter is called
None
Deleter is called


## 6. Read-only and computed properties

In [5]:
class Circle:
    def __init__(self, r):
        self.r = r

    @property
    def area(self):                  # computed on access, no parentheses
        return round(3.14159 * self.r ** 2, 2)

c = Circle(3)
print(c.area)
c.r = 5
print(c.area)
try:
    c.area = 100                     # no setter -> read-only
except AttributeError as e:
    print("AttributeError:", e)

28.27
78.54
AttributeError: property 'area' of 'Circle' object has no setter


## 7. The real descriptor protocol
Any class that defines `__get__`, `__set__` (and optionally `__delete__`) is a descriptor. This lets you **reuse the same validation** for many attributes.

In [6]:
class Validated:
    def __init__(self, kind, max_len=None, low=None, high=None):
        self.kind, self.max_len, self.low, self.high = kind, max_len, low, high

    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"expected {self.kind.__name__}")
        if self.max_len and len(value) > self.max_len:
            raise ValueError(f"max length is {self.max_len}")
        if self.low is not None and not (self.low <= value <= self.high):
            raise ValueError(f"must be between {self.low} and {self.high}")
        setattr(obj, self.name, value)

class Person:
    name = Validated(str, max_len=8)
    age = Validated(int, low=18, high=100)

    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Shubham", 22)
print(p.name, p.age)
for bad in [("Shubham Jadhav", 22), ("sj", 10), (123, 22)]:
    try:
        Person(*bad)
    except (TypeError, ValueError) as e:
        print(bad, "->", type(e).__name__, e)

Shubham 22
('Shubham Jadhav', 22) -> ValueError max length is 8
('sj', 10) -> ValueError must be between 18 and 100
(123, 22) -> TypeError expected str


## 8. Summary
| Tool | When to use |
|---|---|
| `@property` | validation / computed value for **one** attribute |
| custom descriptor class | same validation reused across **many** attributes |